# 03 — Classes, Dataclasses & Enums
### The object-oriented pieces of the project

Most of the project is plain functions and dicts, but three places lean on
classes: `models.py`'s Pydantic models, `retrieval.py`'s `Chunk` (a
`@dataclass`) and `KnowledgeRetriever` (a regular class with state), and
the `Enum` classes that constrain fields like `issue_type` to a fixed set of
valid values.

## 3.1 A class from scratch

A class bundles data (attributes) and behavior (methods) together. `self`
is the instance the method was called on — Python passes it automatically.

In [ ]:
class Ticket:
    def __init__(self, ticket_id: str, text: str):
        self.ticket_id = ticket_id
        self.text = text
        self.resolved = False

    def resolve(self):
        self.resolved = True

    def summary(self) -> str:
        status = "resolved" if self.resolved else "open"
        return f"[{self.ticket_id}] {status}: {self.text[:30]}..."

t = Ticket("T001", "Where is my order? It's been 6 days.")
print(t.summary())
t.resolve()
print(t.summary())


## 3.2 `@dataclass` — classes without the boilerplate

Writing `__init__` by hand for a class that's *just* a bundle of fields
(no custom behavior) is repetitive. `@dataclass` generates `__init__`
(and a few other methods) automatically from type-hinted class attributes.
This is exactly `retrieval.py`'s `Chunk`:

```python
@dataclass
class Chunk:
    source_doc: str
    section: str
    text: str
```

No `__init__` written anywhere — `@dataclass` builds one that takes
`source_doc`, `section`, and `text` as arguments, in that order.

In [ ]:
from dataclasses import dataclass

@dataclass
class Chunk:
    source_doc: str
    section: str
    text: str

c = Chunk(source_doc="refund_policy.md", section="Refund Timelines", text="Refunds take 5-7 days.")
print(c)                 # dataclasses get a readable __repr__ for free
print(c.source_doc, "-", c.section)


## 3.3 A class with real internal state — `KnowledgeRetriever`

`retrieval.py`'s `KnowledgeRetriever` does real work in `__init__`: it loads
files, builds a TF-IDF index, and stores that index as an instance
attribute (`self._matrix`) so `.retrieve()` can reuse it on every call
instead of rebuilding it each time. That's the whole reason a class is used
here instead of a plain function — there's state to build once and reuse.

In [ ]:
class Counter:
    'A tiny example of the same pattern: expensive setup once, cheap calls after.'
    def __init__(self, start: int = 0):
        self._count = start          # leading underscore: "internal, don't touch directly"

    def increment(self) -> int:
        self._count += 1
        return self._count

    @property
    def value(self) -> int:
        return self._count

c = Counter()
c.increment()
c.increment()
c.increment()
print(c.value)   # accessed like an attribute, but it's actually a method (@property)


The `_count` naming (leading underscore) is a convention, not enforced by
Python — it signals "this is internal state, other code shouldn't reach in
and modify it directly." `retrieval.py` uses the same convention:
`self._texts`, `self._vectorizer`, `self._matrix`.

## 3.4 Enums — a fixed set of valid values

`models.py` defines several `Enum` classes. An `Enum` gives you named
constants that are safer than plain strings: `IssueType.FRAUD_SUSPECTED` is
harder to typo than the string `"fraude_suspected"`, and tools/editors can
autocomplete the valid options.

In [ ]:
from enum import Enum

class IssueType(str, Enum):
    ORDER_STATUS = "order_status"
    REFUND_REQUEST = "refund_request"
    FRAUD_SUSPECTED = "fraud_suspected"

print(IssueType.FRAUD_SUSPECTED)
print(IssueType.FRAUD_SUSPECTED.value)     # the actual string value
print(IssueType.FRAUD_SUSPECTED == "fraud_suspected")   # True -- because it inherits from str too


### Why `(str, Enum)` and not just `Enum`

Inheriting from **both** `str` and `Enum` means an `IssueType` member *is
also* a string for **equality and comparison purposes** — it compares equal
to `"fraud_suspected"`, and libraries that check `isinstance(x, str)` (like
some JSON/serialization code) will treat it as one.

**A real gotcha worth knowing:** in Python 3.11+, `str()` and f-strings on a
`(str, Enum)` member print `"IssueType.FRAUD_SUSPECTED"`, not the raw value
`"fraud_suspected"` — even though equality still works. This changed between
Python versions, which is exactly the kind of thing that's confusing until
you've hit it once. If you need the raw string value for display or JSON,
use `.value` explicitly rather than relying on `str()`/f-string conversion.

In [ ]:
print(IssueType.FRAUD_SUSPECTED)                    # -> IssueType.FRAUD_SUSPECTED (not the raw value!)
print(f"{IssueType.FRAUD_SUSPECTED}")               # same thing, via f-string
print(IssueType.FRAUD_SUSPECTED.value)              # -> fraud_suspected -- what you actually want for display
print(IssueType.FRAUD_SUSPECTED == "fraud_suspected")   # -> True -- equality still works regardless


In [ ]:
import json

data = {"issue_type": IssueType.FRAUD_SUSPECTED}
print(json.dumps({"issue_type": IssueType.FRAUD_SUSPECTED.value}))   # .value needed for a clean JSON string either way

# Equality with a plain string still works, which is what escalation_rules.py relies on:
print(IssueType.FRAUD_SUSPECTED == "fraud_suspected")

# But display/logging code should use .value explicitly -- not str()/f-strings -- for the raw value:
print(f"Issue type: {IssueType.FRAUD_SUSPECTED.value}")


## 3.5 Pydantic's `BaseModel` — a class that validates itself

Pydantic's `BaseModel` is a class, same as `Ticket` above, but its
`__init__` is generated for you based on type-hinted class attributes
(similar spirit to `@dataclass`) — with one huge difference: it **validates**
every value against its declared type when you construct it, and raises a
clear error if something doesn't fit.

In [ ]:
# This cell demonstrates the *concept*; running it for real requires
# `pip install pydantic`, which isn't available in every environment.
# The behavior described here is exactly what models.py relies on.

example_code = '''
from pydantic import BaseModel, Field

class TicketClassification(BaseModel):
    ticket_id: str
    confidence: float = Field(ge=0.0, le=1.0)   # must be between 0 and 1

TicketClassification(ticket_id="T001", confidence=0.9)   # OK
TicketClassification(ticket_id="T001", confidence=1.5)   # raises ValidationError -- 1.5 > 1.0
'''
print(example_code)


If you have `pydantic` installed, uncomment and run the cell below to see
the real validation error message.

In [ ]:
# from pydantic import BaseModel, Field
#
# class TicketClassification(BaseModel):
#     ticket_id: str
#     confidence: float = Field(ge=0.0, le=1.0)
#
# TicketClassification(ticket_id="T001", confidence=1.5)


## Exercise

1. Write a `@dataclass` called `Order` with fields `order_id: str`,
   `amount: float`, `status: str`.
2. Write a plain class `OrderBook` with an `__init__` that takes a list of
   `Order` objects and stores it, plus a method `total_value()` that returns
   the sum of all `amount`s. (Hint: `sum(o.amount for o in self.orders)`.)
3. Write an `Enum` called `Urgency` with `LOW`, `MEDIUM`, `HIGH`, `CRITICAL`
   string values, matching the real one in `models.py`. Confirm
   `Urgency.HIGH == "high"` evaluates to `True`.
4. Open `retrieval.py` and find every place `self.` is used inside
   `KnowledgeRetriever`. List each attribute and, in one sentence, what it
   stores.